# LeetCode 434 — Number of Segments in a String bằng Python

**Link đề:** https://leetcode.com/problems/number-of-segments-in-a-string/
**Độ khó:** Easy · **Chủ đề:** String

```python
class Solution:
    def countSegments(self, s: str) -> int:
        ...
```

## Mục tiêu học
1. Hiểu **đoạn** (segment) = dãy liên tiếp ký tự **khác dấu cách**, và các trường hợp biên: chuỗi rỗng, nhiều dấu cách liền nhau, dấu cách ở đầu/cuối.
2. Phân biệt `s.split()` (không đối số) với `s.split(" ")` — bẫy phổ biến nhất của bài.
3. Học kỹ thuật **đếm điểm bắt đầu** (đếm "ranh giới") và **máy trạng thái** (state machine) với cờ `in_segment`.
4. Viết 5 lời giải: `split()` → đếm điểm bắt đầu → máy trạng thái → `zip` cặp ký tự liền kề → regex `\S+`.
5. Trace, chứng minh, Time/Space, lỗi hay gặp, biến thể (đếm từ với nhiều loại khoảng trắng, độ dài từ cuối...).

## 1. Đề bài

Cho chuỗi `s`, trả về **số đoạn** trong chuỗi. Một **đoạn** là một dãy **liên tiếp** các ký tự **không phải dấu cách**.

| `s` | Output | Các đoạn |
|---|:---:|---|
| `"Hello, my name is John"` | `5` | `"Hello,"`, `"my"`, `"name"`, `"is"`, `"John"` |
| `"Hello"` | `1` | `"Hello"` |
| `""` | `0` | (không có) |
| `"   "` | `0` | chỉ có dấu cách |
| `"  a   b  "` | `2` | `"a"`, `"b"` |

**Constraints:**
- `0 <= s.length <= 300` (chuỗi có thể **rỗng**!).
- `s` gồm chữ cái hoa/thường, chữ số, hoặc một trong các ký tự `"!@#$%^&*()_+-=',.:"`.
- Ký tự khoảng trắng **duy nhất** trong `s` là `' '` (không có tab, xuống dòng).

Chú ý: dấu câu như `,` là **một phần** của đoạn (`"Hello,"` là một đoạn).

## 2. Nền tảng Python

### 2.1. Chuỗi, chỉ số, so sánh ký tự

`s[i]` là ký tự thứ `i`. So sánh với dấu cách: `s[i] == " "`. Chuỗi rỗng `""` có `len("") == 0` — vòng `for` trên nó không chạy lần nào.

In [1]:
s = " a,b  c"
for i, ch in enumerate(s):
    print(i, repr(ch), "dấu cách" if ch == " " else "khác dấu cách")
print("len('') =", len(""), "| list(''):", list(""))

0 ' ' dấu cách
1 'a' khác dấu cách
2 ',' khác dấu cách
3 'b' khác dấu cách
4 ' ' dấu cách
5 ' ' dấu cách
6 'c' khác dấu cách
len('') = 0 | list(''): []


### 2.2. `split()` khác `split(" ")`

- `s.split()` (**không đối số**): tách theo **mọi dãy khoảng trắng**, **bỏ** các phần rỗng, bỏ khoảng trắng đầu/cuối.
- `s.split(" ")`: tách tại **từng** dấu cách → hai dấu cách liền nhau tạo ra phần **rỗng** `''`.

In [2]:
s = "  a   b  "
print("split()   :", s.split())
print('split(" "):', s.split(" "))
print('"".split()   :', "".split())
print('"".split(" "):', "".split(" "))     # [''] -> len = 1, không phải 0!

split()   : ['a', 'b']
split(" "): ['', '', 'a', '', '', 'b', '', '']
"".split()   : []
"".split(" "): ['']


### 2.3. `and`, `or` và đánh giá ngắn mạch

`A or B`: nếu `A` đúng thì **không** tính `B`. Nhờ vậy `i == 0 or s[i - 1] == " "` an toàn: khi `i == 0` ta không đọc `s[-1]` (vốn là ký tự **cuối** chuỗi — một bẫy của Python!).

In [3]:
s = "ab"
print("s[-1] =", repr(s[-1]), "(chỉ số âm đọc từ cuối, không báo lỗi)")
i = 0
print("i == 0 or s[i-1] == ' ' ->", i == 0 or s[i - 1] == " ")

s[-1] = 'b' (chỉ số âm đọc từ cuối, không báo lỗi)
i == 0 or s[i-1] == ' ' -> True


### 2.4. `strip()` — bỏ khoảng trắng đầu/cuối

`s.strip()` trả chuỗi mới đã bỏ khoảng trắng ở **hai đầu** (giữa chuỗi giữ nguyên). Một số người viết `s.strip().split(" ")` để "sửa" lỗi đầu/cuối — nhưng nhiều dấu cách **ở giữa** vẫn sinh phần rỗng, và chuỗi toàn dấu cách vẫn cho `['']`. Hãy tự kiểm tra:

In [4]:
for s in ["  a b  ", "a   b", "   "]:
    t = s.strip()
    print(f"{s!r:<10} strip -> {t!r:<8} strip().split(' ') -> {t.split(' ')}  (len={len(t.split(' '))})")

'  a b  '  strip -> 'a b'    strip().split(' ') -> ['a', 'b']  (len=2)
'a   b'    strip -> 'a   b'  strip().split(' ') -> ['a', '', '', 'b']  (len=4)
'   '      strip -> ''       strip().split(' ') -> ['']  (len=1)


## 3. Cách trực giác (baseline) — `split()`

Python đã có sẵn công cụ tách từ đúng nghĩa đề: `len(s.split())`. Ngắn, đúng, và là cách nên dùng trong thực tế. Tuy vậy phỏng vấn thường yêu cầu **tự cài đặt** — và `split()` tạo ra list các chuỗi con, tốn `O(n)` bộ nhớ chỉ để **đếm**.

In [5]:
def count_segments_split(s: str) -> int:
    return len(s.split())

for s in ["Hello, my name is John", "Hello", "", "   ", "  a   b  "]:
    print(repr(s), "->", count_segments_split(s))

'Hello, my name is John' -> 5
'Hello' -> 1
'' -> 0
'   ' -> 0
'  a   b  ' -> 2


## 4. Nút thắt và insight

**Nút thắt:** muốn đếm mà không tạo list chuỗi con; và phải xử lý đúng nhiều dấu cách liền nhau, đầu/cuối, chuỗi rỗng.

**Insight — đếm điểm bắt đầu:** mỗi đoạn có **đúng một** ký tự đầu tiên. Ký tự `s[i]` là **điểm bắt đầu** của một đoạn khi:
1. `s[i]` **không** phải dấu cách, và
2. nó đứng **đầu chuỗi** (`i == 0`) **hoặc** ký tự trước nó **là** dấu cách.

Số đoạn = số điểm bắt đầu. Không cần biết đoạn dài bao nhiêu!

```text
s = "  a   bc  "
     0123456789
         ^     ^       điểm bắt đầu: i=2 ('a'), i=6 ('b')
'c' ở i=7 không phải điểm bắt đầu vì s[6] = 'b' không phải dấu cách
```

In [6]:
s = "  a   bc  "
starts = [i for i in range(len(s)) if s[i] != " " and (i == 0 or s[i - 1] == " ")]
print("các điểm bắt đầu:", starts, "->", [s[i] for i in starts], "-> số đoạn =", len(starts))

các điểm bắt đầu: [2, 6] -> ['a', 'b'] -> số đoạn = 2


## 5. Lời giải chính — đếm điểm bắt đầu

**Đoạn 1:** `count = 0`.

**Đoạn 2:** duyệt `i` từ `0` đến `len(s) - 1`:
```python
if s[i] != " " and (i == 0 or s[i - 1] == " "):
    count += 1
```
**Đoạn 3:** `return count`.

Chuỗi rỗng: vòng lặp không chạy → `0`. Chuỗi toàn dấu cách: điều kiện 1 luôn sai → `0`.

In [7]:
def count_segments_starts(s: str) -> int:
    count = 0
    for i in range(len(s)):
        if s[i] != " " and (i == 0 or s[i - 1] == " "):
            count += 1
    return count

for s in ["Hello, my name is John", "Hello", "", "   ", "  a   b  "]:
    print(repr(s), "->", count_segments_starts(s))

'Hello, my name is John' -> 5
'Hello' -> 1
'' -> 0
'   ' -> 0
'  a   b  ' -> 2


## 6. TRACE từng bước

Trace `s = " ab  c"`:

| i | s[i] | khác ' '? | i==0 hoặc s[i-1]==' '? | Bắt đầu đoạn? | count |
|:---:|:---:|:---:|:---:|:---:|:---:|
| 0 | `' '` | không | — | không | 0 |
| 1 | `a` | có | `s[0]=' '` → có | **có** | 1 |
| 2 | `b` | có | `s[1]='a'` → không | không | 1 |
| 3 | `' '` | không | — | không | 1 |
| 4 | `' '` | không | — | không | 1 |
| 5 | `c` | có | `s[4]=' '` → có | **có** | 2 |

In [8]:
def trace_count_segments(s: str) -> int:
    count = 0
    print(f"s = {s!r}")
    for i in range(len(s)):
        not_space = s[i] != " "
        prev_ok = (i == 0 or s[i - 1] == " ")
        is_start = not_space and prev_ok
        if is_start:
            count += 1
        prev = "đầu chuỗi" if i == 0 else repr(s[i - 1])
        print(f"  i={i}: s[i]={s[i]!r:<4} khác cách={not_space!s:<5} trước={prev:<10} "
              f"bắt đầu={is_start!s:<5} count={count}")
    print("  kết quả:", count)
    return count

trace_count_segments(" ab  c")
print()
trace_count_segments("x")

s = ' ab  c'
  i=0: s[i]=' '  khác cách=False trước=đầu chuỗi  bắt đầu=False count=0
  i=1: s[i]='a'  khác cách=True  trước=' '        bắt đầu=True  count=1
  i=2: s[i]='b'  khác cách=True  trước='a'        bắt đầu=False count=1
  i=3: s[i]=' '  khác cách=False trước='b'        bắt đầu=False count=1
  i=4: s[i]=' '  khác cách=False trước=' '        bắt đầu=False count=1
  i=5: s[i]='c'  khác cách=True  trước=' '        bắt đầu=True  count=2
  kết quả: 2

s = 'x'
  i=0: s[i]='x'  khác cách=True  trước=đầu chuỗi  bắt đầu=True  count=1
  kết quả: 1


1

## 7. Các lời giải khác

### 7.1. Cách 3 — máy trạng thái (state machine)

Giữ một cờ `in_segment` = "đang ở trong một đoạn?". Duyệt từng ký tự:
- Gặp ký tự thường mà `in_segment == False` → **vừa bước vào** đoạn mới: `count += 1`, `in_segment = True`.
- Gặp dấu cách → `in_segment = False`.

Cách này không cần nhìn lại `s[i - 1]` — hữu ích khi đọc dữ liệu dạng **luồng** (stream), từng ký tự một.

In [9]:
def count_segments_state(s: str) -> int:
    count = 0
    in_segment = False
    for ch in s:
        if ch == " ":
            in_segment = False
        elif not in_segment:
            in_segment = True
            count += 1
    return count

for s in ["Hello, my name is John", "", "   ", "  a   b  "]:
    print(repr(s), "->", count_segments_state(s))

'Hello, my name is John' -> 5
'' -> 0
'   ' -> 0
'  a   b  ' -> 2


### 7.2. Cách 4 — đếm cặp liền kề bằng `zip`

Thêm một dấu cách "giả" vào đầu: `" " + s`. Khi đó điểm bắt đầu đoạn là mọi cặp liền kề `(trước, sau)` có `trước == " "` và `sau != " "`. `zip(a, b)` ghép từng cặp phần tử cùng vị trí; `zip(t, t[1:])` cho các cặp liền kề.

In [10]:
t = " " + "ab c"
print(list(zip(t, t[1:])))

def count_segments_zip(s: str) -> int:
    t = " " + s
    return sum(1 for prev, cur in zip(t, t[1:]) if prev == " " and cur != " ")

for s in ["Hello, my name is John", "", "   ", "  a   b  "]:
    print(repr(s), "->", count_segments_zip(s))

[(' ', 'a'), ('a', 'b'), ('b', ' '), (' ', 'c')]
'Hello, my name is John' -> 5
'' -> 0
'   ' -> 0
'  a   b  ' -> 2


### 7.3. Cách 5 — biểu thức chính quy `\S+`

Trong regex, `\S` là "một ký tự **không** phải khoảng trắng", `+` là "một hoặc nhiều". `re.findall(r"\S+", s)` trả mọi đoạn. Lưu ý `\S` coi tab/xuống dòng cũng là khoảng trắng — với đề này (chỉ có `' '`) thì kết quả như nhau. Có thể viết chính xác hơn bằng `[^ ]+` (không phải dấu cách).

In [11]:
import re

print(re.findall(r"\S+", "Hello, my name is John"))
print(re.findall(r"[^ ]+", "  a   b  "))

def count_segments_regex(s: str) -> int:
    return len(re.findall(r"[^ ]+", s))

for s in ["Hello, my name is John", "", "   ", "  a   b  "]:
    print(repr(s), "->", count_segments_regex(s))

['Hello,', 'my', 'name', 'is', 'John']
['a', 'b']
'Hello, my name is John' -> 5
'' -> 0
'   ' -> 0
'  a   b  ' -> 2


## 8. Code nộp LeetCode

Bản chính: đếm điểm bắt đầu (`O(1)` bộ nhớ). Bản phụ: một dòng `split()` — đúng và được chấp nhận.

In [12]:
class Solution:
    def countSegments(self, s: str) -> int:
        count = 0
        for i in range(len(s)):
            if s[i] != " " and (i == 0 or s[i - 1] == " "):
                count += 1
        return count


sol = Solution()
print(sol.countSegments("Hello, my name is John"))  # 5
print(sol.countSegments("Hello"))                   # 1

5
1


In [13]:
class SolutionSplit:
    def countSegments(self, s: str) -> int:
        return len(s.split())

print(SolutionSplit().countSegments("Hello, my name is John"), SolutionSplit().countSegments(""))

5 0


## 9. Vì sao đúng?

**Song ánh** giữa đoạn và điểm bắt đầu:
- Mỗi đoạn là dãy tối đại `s[a..b]` không chứa dấu cách. "Tối đại" nghĩa là `a == 0` hoặc `s[a-1] == ' '`. Do đó `s[a]` thỏa điều kiện "điểm bắt đầu" → mỗi đoạn có **ít nhất** một điểm bắt đầu.
- Ngược lại, nếu `s[i]` là điểm bắt đầu thì đoạn chứa `s[i]` bắt đầu đúng tại `i` (vì bên trái là đầu chuỗi hoặc dấu cách). Một đoạn chỉ có **một** vị trí đầu → mỗi đoạn có **đúng một** điểm bắt đầu.
- Hai đoạn khác nhau có điểm bắt đầu khác nhau.

⇒ số điểm bắt đầu = số đoạn. ∎

**Máy trạng thái:** bất biến "`in_segment` đúng ⟺ ký tự vừa đọc là ký tự khác dấu cách" → `count += 1` xảy ra đúng khi chuyển từ "ngoài" sang "trong" = điểm bắt đầu.

## 10. Độ phức tạp

Gọi `n = len(s)` (`n <= 300`).

| Cách | Time | Space phụ |
|---|:---:|:---:|
| `split()` | `O(n)` | `O(n)` (list các đoạn) |
| **Đếm điểm bắt đầu** | `O(n)` | `O(1)` |
| Máy trạng thái | `O(n)` | `O(1)` |
| `zip` cặp liền kề | `O(n)` | `O(n)` (tạo `t` và `t[1:]`) |
| Regex | `O(n)` | `O(n)` (list kết quả) |

Mọi cách phải đọc mỗi ký tự ít nhất một lần ⇒ `O(n)` là tối ưu.

## 11. Bộ test

In [14]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS  {name:<34} -> {actual}")

sol = Solution()
cases = [
    ("đề 1", "Hello, my name is John", 5),
    ("đề 2", "Hello", 1),
    ("chuỗi rỗng", "", 0),
    ("một dấu cách", " ", 0),
    ("toàn dấu cách", "     ", 0),
    ("cách đầu/cuối", "  hi  ", 1),
    ("nhiều cách giữa", "a    b", 2),
    ("một ký tự", "x", 1),
    ("dấu câu là một đoạn", ", . !", 3),
    ("ký tự đặc biệt", "love live! mu'sic forever", 4),
    ("dài 300 xen kẽ", "a " * 150, 150),
]
for name, s, exp in cases:
    check(name, sol.countSegments(s), exp)

PASS  đề 1                               -> 5
PASS  đề 2                               -> 1
PASS  chuỗi rỗng                         -> 0
PASS  một dấu cách                       -> 0
PASS  toàn dấu cách                      -> 0
PASS  cách đầu/cuối                      -> 1
PASS  nhiều cách giữa                    -> 2
PASS  một ký tự                          -> 1
PASS  dấu câu là một đoạn                -> 3
PASS  ký tự đặc biệt                     -> 4
PASS  dài 300 xen kẽ                     -> 150


In [15]:
import random
rng = random.Random(434)
funcs = [count_segments_starts, count_segments_state, count_segments_zip, count_segments_regex,
         Solution().countSegments, SolutionSplit().countSegments]
alphabet = "  ab,.!'"
for _ in range(3000):
    s = "".join(rng.choice(alphabet) for _ in range(rng.randint(0, 30)))
    exp = count_segments_split(s)
    for f in funcs:
        assert f(s) == exp, (f.__name__, s)
print("3000 chuỗi ngẫu nhiên (nhiều dấu cách): 6 cách khớp split() -> PASS")

3000 chuỗi ngẫu nhiên (nhiều dấu cách): 6 cách khớp split() -> PASS


## 12. Lỗi người mới hay gặp

### Lỗi 1: Dùng `split(" ")`
```python
return len(s.split(" "))
```
`""` → 1, `"a  b"` → 3, `" a"` → 2. Phần rỗng `''` bị đếm.

### Lỗi 2: Đếm dấu cách rồi cộng 1
```python
return s.count(" ") + 1
```
Chỉ đúng khi các từ cách nhau **đúng một** dấu cách và không có cách đầu/cuối.

In [16]:
def wrong_split_space(s):
    return len(s.split(" "))

def wrong_count_spaces(s):
    return s.count(" ") + 1

for s in ["", "a  b", " a", "Hello, my name is John"]:
    print(f"{s!r:<26} split(' '): {wrong_split_space(s)} | count+1: {wrong_count_spaces(s)} | đúng: {sol.countSegments(s)}")

''                         split(' '): 1 | count+1: 1 | đúng: 0
'a  b'                     split(' '): 3 | count+1: 3 | đúng: 2
' a'                       split(' '): 2 | count+1: 2 | đúng: 1
'Hello, my name is John'   split(' '): 5 | count+1: 5 | đúng: 5


### Lỗi 3: Dùng `s[i - 1]` khi `i == 0` mà không chặn
`s[-1]` là ký tự **cuối**! Với `"a b "`, tại `i = 0` code sẽ xét `s[-1] = ' '` → may mắn đúng; nhưng với `"ab"` → `s[-1] = 'b'` → `'a'` không được tính là điểm bắt đầu → kết quả 0, sai.

### Lỗi 4: Đếm điểm **kết thúc** nhưng quên đoạn cuối chuỗi
Đếm "ký tự thường mà sau nó là dấu cách" → đoạn cuối (không có dấu cách theo sau) bị bỏ sót.

In [17]:
def wrong_no_i0_guard(s):
    return sum(1 for i in range(len(s)) if s[i] != " " and s[i - 1] == " ")

def wrong_count_ends(s):
    return sum(1 for i in range(len(s) - 1) if s[i] != " " and s[i + 1] == " ")

print("lỗi 3 'ab':", wrong_no_i0_guard("ab"), "(đúng 1)")
print("lỗi 3 'a b ':", wrong_no_i0_guard("a b "), "(đúng 2, nhưng chỉ may mắn)")
print("lỗi 4 'a b':", wrong_count_ends("a b"), "(đúng 2)")

lỗi 3 'ab': 0 (đúng 1)
lỗi 3 'a b ': 2 (đúng 2, nhưng chỉ may mắn)
lỗi 4 'a b': 1 (đúng 2)


### Lỗi 5: Quên chuỗi rỗng
Viết `if s[0] != " "` trước vòng lặp → `IndexError` với `s = ""`. Constraints cho phép `s.length = 0`.

In [18]:
try:
    empty = ""
    print(empty[0])
except IndexError as e:
    print("IndexError:", e)

IndexError: string index out of range


## 13. Mẹo nhận dạng pattern

- "Đếm số **nhóm/khối/đoạn** liên tiếp thỏa tính chất" → đếm **điểm bắt đầu** (phần tử thỏa và phần tử trước không thỏa) hoặc dùng **cờ trạng thái**.
- Cùng ý tưởng: đếm số "đảo" 1 trong mảng nhị phân 1 chiều, số nhóm ký tự giống nhau liên tiếp, số lần chuyển trạng thái.
- Có chữ "khoảng trắng/space" → nghĩ ngay tới bẫy `split(" ")` vs `split()`, cách đầu/cuối, chuỗi rỗng.

Bài liên quan: 58 Length of Last Word, 151 Reverse Words in a String, 1446 Consecutive Characters, 485 Max Consecutive Ones.

## 14. Biến thể

1. **Độ dài đoạn cuối** (LeetCode 58).
2. **Đoạn dài nhất** — máy trạng thái thêm biến độ dài hiện tại.
3. **Đếm khối số 1 liên tiếp** trong mảng `[0,1,1,0,1]` → 2.

In [19]:
def length_of_last_word(s: str) -> int:
    i = len(s) - 1
    while i >= 0 and s[i] == " ":
        i -= 1
    length = 0
    while i >= 0 and s[i] != " ":
        length += 1
        i -= 1
    return length

def count_blocks_of_ones(arr: list) -> int:
    return sum(1 for i, x in enumerate(arr) if x == 1 and (i == 0 or arr[i - 1] == 0))

print(length_of_last_word("   fly me   to   the moon  "))   # 4
print(count_blocks_of_ones([0, 1, 1, 0, 1]), count_blocks_of_ones([1, 1, 1]))

4
2 1


## 15. Bài tập (TODO)

**Bài 1 (tính tay):** Đánh dấu điểm bắt đầu trong `"  to be,or  not "`. Có bao nhiêu đoạn?

**Bài 2 (đoán output):** `len(" a b ".split(" "))` và `len(" a b ".split())`?

**Bài 3 (điền chỗ trống):** hoàn thiện máy trạng thái.

In [20]:
def exercise_count_segments(s: str) -> int:
    count = 0
    in_segment = False
    for ch in s:
        # TODO 1: nếu ch là dấu cách -> ra khỏi đoạn
        # TODO 2: nếu không và chưa ở trong đoạn -> vào đoạn, tăng count
        pass
    return count

# assert exercise_count_segments("Hello, my name is John") == 5
# assert exercise_count_segments("   ") == 0

**Bài 4 (tự viết):** `exercise_longest_segment(s)` trả độ dài đoạn dài nhất (0 nếu không có), một lần duyệt, không dùng `split`.

**Bài 5 (biến thể):** `exercise_segments_positions(s)` trả list các cặp `(start, end)` (end không bao gồm) của từng đoạn.

**Bài 6 (mini challenge):** `exercise_count_segments_multi(s, seps)` — đếm đoạn khi **nhiều** ký tự được coi là phân cách (ví dụ `seps = " ,;"`).

In [21]:
def exercise_longest_segment(s: str) -> int:
    # TODO
    pass

def exercise_segments_positions(s: str) -> list:
    # TODO
    pass

def exercise_count_segments_multi(s: str, seps: str) -> int:
    # TODO
    pass

# assert exercise_longest_segment("a bbb  cc") == 3
# assert exercise_segments_positions(" ab c") == [(1, 3), (4, 5)]
# assert exercise_count_segments_multi("a,b;;c d", " ,;") == 4

## 16. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | Tìm ký tự sau dấu cách | `,` thuộc đoạn `be,or` | Đoạn: `to`, `be,or`, `not` |
| 2 | `split(" ")` giữ phần rỗng | Đầu và cuối đều có `''` | `['', 'a', 'b', '']` |
| 3 | Theo mục 7.1 | `if ch == " ": in_segment = False` | `elif not in_segment: in_segment = True; count += 1` |
| 4 | Máy trạng thái + độ dài | `cur += 1` khi trong đoạn, reset khi gặp cách | `best = max(best, cur)` |
| 5 | Ghi lại vị trí bắt đầu | Khi kết thúc đoạn (gặp cách hoặc hết chuỗi) thì append | Nhớ xử lý đoạn cuối sau vòng lặp |
| 6 | Thay `== " "` bằng `in seps` | Dùng `set(seps)` | `if ch in sep_set: ...` |

## 17. Đáp án tham khảo

- **Bài 1:** `"  to be,or  not "` → điểm bắt đầu tại `t` (i=2), `b` (i=5), `n` (i=12) → **3** đoạn.
- **Bài 2:** `" a b ".split(" ")` = `['', 'a', 'b', '']` → **4**; `split()` = `['a', 'b']` → **2**.

In [22]:
trace_count_segments("  to be,or  not ")
print("Bài 2:", len(" a b ".split(" ")), len(" a b ".split()))

s = '  to be,or  not '
  i=0: s[i]=' '  khác cách=False trước=đầu chuỗi  bắt đầu=False count=0
  i=1: s[i]=' '  khác cách=False trước=' '        bắt đầu=False count=0
  i=2: s[i]='t'  khác cách=True  trước=' '        bắt đầu=True  count=1
  i=3: s[i]='o'  khác cách=True  trước='t'        bắt đầu=False count=1
  i=4: s[i]=' '  khác cách=False trước='o'        bắt đầu=False count=1
  i=5: s[i]='b'  khác cách=True  trước=' '        bắt đầu=True  count=2
  i=6: s[i]='e'  khác cách=True  trước='b'        bắt đầu=False count=2
  i=7: s[i]=','  khác cách=True  trước='e'        bắt đầu=False count=2
  i=8: s[i]='o'  khác cách=True  trước=','        bắt đầu=False count=2
  i=9: s[i]='r'  khác cách=True  trước='o'        bắt đầu=False count=2
  i=10: s[i]=' '  khác cách=False trước='r'        bắt đầu=False count=2
  i=11: s[i]=' '  khác cách=False trước=' '        bắt đầu=False count=2
  i=12: s[i]='n'  khác cách=True  trước=' '        bắt đầu=True  count=3
  i=13: s[i]='o'  khác cách=True  trướ

In [23]:
def answer_count_segments(s: str) -> int:
    count, in_segment = 0, False
    for ch in s:
        if ch == " ":
            in_segment = False
        elif not in_segment:
            in_segment = True
            count += 1
    return count

def answer_longest_segment(s: str) -> int:
    best = cur = 0
    for ch in s:
        if ch == " ":
            cur = 0
        else:
            cur += 1
            best = max(best, cur)
    return best

def answer_segments_positions(s: str) -> list:
    out = []
    start = None
    for i, ch in enumerate(s):
        if ch != " " and start is None:
            start = i
        elif ch == " " and start is not None:
            out.append((start, i))
            start = None
    if start is not None:
        out.append((start, len(s)))
    return out

def answer_count_segments_multi(s: str, seps: str) -> int:
    sep_set = set(seps)
    count, inside = 0, False
    for ch in s:
        if ch in sep_set:
            inside = False
        elif not inside:
            inside = True
            count += 1
    return count

assert answer_count_segments("Hello, my name is John") == 5 and answer_count_segments("   ") == 0
assert answer_longest_segment("a bbb  cc") == 3 and answer_longest_segment("") == 0
assert answer_segments_positions(" ab c") == [(1, 3), (4, 5)] and answer_segments_positions("") == []
assert answer_count_segments_multi("a,b;;c d", " ,;") == 4
assert answer_count_segments_multi("a b", " ") == Solution().countSegments("a b")
print("Đáp án bài 3–6 PASS")

Đáp án bài 3–6 PASS


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def countSegments(self, s: str) -> int:
        count = 0
        for i in range(len(s)):
            # điểm bắt đầu: không phải cách, và (đầu chuỗi hoặc trước là cách)
            if s[i] != " " and (i == 0 or s[i - 1] == " "):
                count += 1
        return count
# Hoặc đơn giản: return len(s.split())     # KHÔNG phải split(" ")
```

**Checklist:**
- [ ] Chuỗi rỗng `""` → 0 (không đọc `s[0]`).
- [ ] Nhiều dấu cách liền nhau, đầu/cuối → không tạo đoạn rỗng.
- [ ] Không dùng `split(" ")`, không `count(" ") + 1`.
- [ ] Chặn `i == 0` trước khi đọc `s[i - 1]`.
- [ ] Dấu câu là một phần của đoạn.
- [ ] Time `O(n)`, Space `O(1)`.